# Credit Risk Pipeline - End-to-End ML Project

**Objective**: Build a production-grade ML pipeline for credit default prediction.

**Dataset**: German Credit (UCI/OpenML) - 1000 borrowers, 20 features

> Part of [AIML Companion](https://github.com/genieincodebottle/aiml-companion) - ML Pipeline Track

## 1. Setup & Imports

In [ ]:
# Install dependencies (run this cell first on Kaggle/Colab)
!pip install -q shap scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

print("All imports successful!")

## 2. Data Loading

In [ ]:
try:
    data = fetch_openml(data_id=31, as_frame=True, parser="auto")
    df = data.frame.copy()
except Exception as e:
    print(f"OpenML fetch failed ({e}). Falling back to CSV URL...")
    url = "https://raw.githubusercontent.com/genieincodebottle/aiml-companion/main/projects/credit-risk-pipeline/data/german_credit.csv"
    try:
        df = pd.read_csv(url)
    except Exception:
        raise RuntimeError(
            "Could not load data from OpenML or fallback URL.\n"
            "Manual fix: download German Credit from https://www.openml.org/d/31 "
            "and place as 'german_credit.csv' in the notebook directory, then use:\n"
            "  df = pd.read_csv('german_credit.csv')"
        )

if "class" in df.columns:
    df["target"] = (df["class"] == "bad").astype(int)
    df = df.drop(columns=["class"])

print(f"Shape: {df.shape}")
print(f"Default rate: {df['target'].mean():.1%}")
df.head()

In [ ]:
df.info()

## 3. Data Quality Assessment

In [ ]:
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nTarget distribution:")
print(df["target"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["target"].value_counts()
ax.bar(["No Default (0)", "Default (1)"], counts.values, color=["#22c55e", "#ef4444"])
for i, c in enumerate(counts.values):
    ax.text(i, c + 10, f"{c} ({c/len(df):.1%})", ha="center")
ax.set_title("Target Distribution")
plt.tight_layout()
plt.show()

## 4. EDA

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "target"]
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
corr = df[numeric_cols + ["target"]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
n = min(len(numeric_cols), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(numeric_cols[:n]):
    ax = axes[i//3][i%3]
    for val, color, label in [(0, "#22c55e", "No Default"), (1, "#ef4444", "Default")]:
        ax.hist(df[df["target"]==val][col].dropna(), bins=25, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)
for i in range(n, 6):
    axes[i//3][i%3].set_visible(False)
plt.suptitle("Feature Distributions by Target", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
if "credit_amount" in df.columns and "duration" in df.columns:
    df["monthly_burden"] = df["credit_amount"] / df["duration"].replace(0, np.nan)
    df["loan_burden"] = (df["credit_amount"] * df["duration"]) / df["credit_amount"].median()
    print("Added monthly_burden, loan_burden")

if "age" in df.columns:
    df["age_group"] = pd.cut(df["age"], bins=[0,25,35,45,55,100], labels=["18-25","26-35","36-45","46-55","55+"])
    print(f"Age groups: {df['age_group'].value_counts().sort_index().to_dict()}")

for col in numeric_cols:
    if abs(df[col].skew()) > 2.0:
        df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
        print(f"Added log_{col} (skew={df[col].skew():.1f})")

print(f"\nTotal features: {df.shape[1]}")

## 6. Preprocessing Pipeline

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", KNNImputer(n_neighbors=5)), ("scaler", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
], remainder="drop")

print(f"Numeric: {len(numeric_features)} features -> KNNImputer -> StandardScaler")
print(f"Categorical: {len(categorical_features)} features -> OneHotEncoder")

## 7. Model Training (5-Fold CV)

In [ ]:
lr_pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42))])
gbc_pipe = Pipeline([("pre", preprocessor), ("clf", GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, subsample=0.8, random_state=42))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, pipe in [("LogisticRegression", lr_pipe), ("GradientBoosting", gbc_pipe)]:
    print(f"\nTraining {name}...")
    scores = cross_validate(pipe, X, y, cv=cv, scoring=["roc_auc", "recall", "f1"])
    results[name] = {k: scores[f"test_{k}"].mean() for k in ["roc_auc", "recall", "f1"]}
    results[name]["roc_auc_std"] = scores["test_roc_auc"].std()
    print(f"  ROC-AUC: {results[name]['roc_auc']:.3f} (+/- {results[name]['roc_auc_std']:.3f})")
    print(f"  Recall:  {results[name]['recall']:.3f}")
    print(f"  F1:      {results[name]['f1']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
w = 0.35
lr = [results["LogisticRegression"][m] for m in ["roc_auc", "recall", "f1"]]
gbc = [results["GradientBoosting"][m] for m in ["roc_auc", "recall", "f1"]]
ax.bar(x-w/2, lr, w, label="Logistic Regression", color="#3b82f6")
ax.bar(x+w/2, gbc, w, label="Gradient Boosting", color="#f97316")
ax.set_xticks(x)
ax.set_xticklabels(["ROC-AUC", "Recall", "F1"])
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Model Comparison")
plt.tight_layout()
plt.show()

## 8. Cost-Sensitive Threshold Tuning

FN cost = $10 (missed default), FP cost = $1 (denied good borrower)

> **Note**: For simplicity, we fit on the full dataset and tune the threshold here. In production, always tune the threshold on a held-out validation set or use nested cross-validation to avoid overfitting the threshold to training data.

In [ ]:
best_name = max(results, key=lambda k: results[k]["roc_auc"])
best_pipe = lr_pipe if best_name == "LogisticRegression" else gbc_pipe
best_pipe.fit(X, y)
y_proba = best_pipe.predict_proba(X)[:, 1]

COST_FN, COST_FP = 10, 1
thresholds = np.arange(0.05, 0.95, 0.01)
costs = []
for t in thresholds:
    cm = confusion_matrix(y, (y_proba>=t).astype(int))
    costs.append(cm[1,0]*COST_FN + cm[0,1]*COST_FP)

opt_t = thresholds[np.argmin(costs)]
min_cost = min(costs)
print(f"Best model: {best_name}")
print(f"Optimal threshold: {opt_t:.2f}")
print(f"Minimum cost: ${min_cost:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, costs, "b-", lw=2)
axes[0].axvline(x=opt_t, color="red", linestyle="--", label=f"Optimal: {opt_t:.2f}")
axes[0].axvline(x=0.5, color="gray", linestyle=":", label="Default: 0.50")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Total Cost ($)")
axes[0].set_title("Cost-Sensitive Threshold Tuning")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

precision, recall, pr_t = precision_recall_curve(y, y_proba)
axes[1].plot(recall, precision, "b-", lw=2)
idx = np.argmin(np.abs(pr_t - opt_t))
axes[1].plot(recall[idx], precision[idx], "ro", markersize=12, label=f"Optimal (t={opt_t:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
y_def = (y_proba >= 0.5).astype(int)
y_opt = (y_proba >= opt_t).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, yp, title in [(axes[0], y_def, "Threshold=0.50"), (axes[1], y_opt, f"Threshold={opt_t:.2f}")]:
    cm = confusion_matrix(y, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Default","Default"], yticklabels=["No Default","Default"], ax=ax)
    cost = cm[1,0]*COST_FN + cm[0,1]*COST_FP
    ax.set_title(f"{title} (Cost=${cost:,})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.suptitle(f"Confusion Matrix - {best_name}", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nClassification Report (threshold={opt_t:.2f}):")
print(classification_report(y, y_opt, target_names=["No Default", "Default"]))

## 9. SHAP Explanations

In [ ]:
try:
    import shap
    pre = best_pipe.named_steps["pre"]
    clf = best_pipe.named_steps["clf"]
    X_t = pre.transform(X)
    feat_names = pre.get_feature_names_out()
    explainer = shap.Explainer(clf, X_t[:200])
    sv = explainer(X_t[:200])
    # Handle 3D SHAP values (some models return shape [n_samples, n_features, n_classes])
    vals = sv.values
    if vals.ndim == 3:
        vals = vals[:, :, 1]  # take positive class (default)
    imp = dict(zip(feat_names, np.abs(vals).mean(axis=0)))
    imp = dict(sorted(imp.items(), key=lambda x: x[1], reverse=True))
    top = list(imp.keys())[:15]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(top)), [imp[f] for f in top][::-1], color="#3b82f6")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top[::-1], fontsize=9)
    ax.set_xlabel("Mean |SHAP|")
    ax.set_title("Top 15 Feature Importance")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("SHAP not installed. Run: pip install shap")
except Exception as e:
    print(f"SHAP failed: {e}\nThis can happen with certain model/SHAP version combinations.")

## 10. PSI Drift Monitoring

In [ ]:
def compute_psi(ref, cur, n_bins=10):
    ref, cur = np.array(ref, dtype=float), np.array(cur, dtype=float)
    ref, cur = ref[~np.isnan(ref)], cur[~np.isnan(cur)]
    if len(ref)==0 or len(cur)==0: return 0.0
    bp = np.unique(np.percentile(ref, np.linspace(0,100,n_bins+1)))
    if len(bp)<2: return 0.0
    eps = 1e-4
    r = np.histogram(ref, bins=bp)[0]/len(ref)+eps
    c = np.histogram(cur, bins=bp)[0]/len(cur)+eps
    r, c = r/r.sum(), c/c.sum()
    return float(np.sum((c-r)*np.log(c/r)))

np.random.seed(42)
ref_data = X[numeric_features].copy()
prod_data = ref_data.sample(n=200, replace=True).copy()
if "credit_amount" in prod_data.columns:
    prod_data["credit_amount"] = prod_data["credit_amount"] * 1.5 + 500

print(f"Feature                       PSI  Status")
print("-"*55)
for col in numeric_features:
    if col in prod_data.columns:
        psi = compute_psi(ref_data[col].values, prod_data[col].values)
        status = "OK" if psi<0.10 else "WARN" if psi<0.25 else "ALERT"
        print(f"{col:<25} {psi:>8.4f} {status}")

## Summary

1. **Data pipeline** - German Credit loaded, cleaned, validated
2. **EDA** - Distributions, correlations, target segmentation
3. **Feature engineering** - Monthly burden, loan burden, age buckets, log transforms
4. **Model training** - LR + GBC with sklearn Pipeline (leakage-proof)
5. **Cost-sensitive evaluation** - 10:1 FN/FP threshold tuning
6. **SHAP explanations** - Feature importance for adverse action reasons
7. **PSI monitoring** - Drift detection for production

> Full project: [github.com/genieincodebottle/aiml-companion/projects/credit-risk-pipeline](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/credit-risk-pipeline)